# RPCA-based Risk Modeling for Portfolio Optimization


## An Unsupervised, Spectral ML Approach

Какие-нибудь тексты

# I. Формулы и описание Linear constrained quadratic optimization

Формулы из учебника


# II. Графики и визуализации

- вставить график "/home/dimitri/Social-processes/Make-Up/weight_change_l1_norm_timeseries_200.png" из папки "/home/dimitri/Social-processes/Make-Up/"
- показать cond_number = **np.linalg.cond(Sigma)**  # condition number of Σ

<img src="../Make-Up/weight_change_l1_norm_timeseries_200.png" alt="Изменение веса L1 нормы" style="width:20%; height:auto;"/>



In [17]:
print("Condition number of Σ:", 40000)

Condition number of Σ: 40000


# Центровка по столбцам

- средние по столбцам за окно почти никогда не равны нулю точно (есть остаточный дрейф/смещение, особенно на коротких окнах).
- В RPCA постоянные смещения по столбцу дают лишний ранг-1 вклад и ухудшают контроль ранга низкоранговой части 𝐿.
- Робастная центровка (лучше медиана, чем среднее) делает модель более устойчивой, а разрежённая часть 𝑆 начинает «ловить» именно всплески/аномалии вокруг нулевой базы.

In [18]:
%run robust_PCA_step1.py

Sanity — |column median| after centering (should be ~0):
count    189.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
dtype: float64

[INFO] Step 1 (centering). Files saved:
 - returns_centered.parquet
 - center_shift_vector.csv


# Масштабирование по столбцам

## Почему:
- RPCA штрафует элементы матрицы равномерно; если волатильности активов сильно различаются (а в реальности так и есть), то без масштабирования активы с большой дисперсией «доминируют» в штрафах, и разрежённость 𝑆/ранг 𝐿 подстраиваются под масштаб, а не под структуру.
- Приведение столбцов к единому масштабу делает штрафы сопоставимыми между активами и улучшает восстановление низко ранговой структуры.
## Чем масштабировать:
- Лучше робастной шкалой: MAD (median absolute deviation). MAD (Median Absolute Deviation)  — это робастная мера “размаха” (масштаба) данных. Считается как медиана абсолютных отклонений наблюдений от робастного центра (обычно от медианы столбца):

MAD = median(|xᵢ − median(x)|).

В отличие от стандартного отклонения, MAD почти не “ломается” на выбросах и тяжёлых хвостах, поэтому подходит для финансовых рядов.

In [19]:
%run robust_PCA_step2.py


Sanity — robust spread (MAD*1.4826) after scaling (~1 expected):
count    189.0
mean       inf
std        NaN
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        inf
dtype: float64

[INFO] Step 2 (scaling) completed. Files saved:
 - returns_centered_scaled.parquet
 - scale_vector.csv


/home/dimitri/.local/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


## Что мы получили после
scaled_df, scale_vec = scale_columns_mad(centered_df, mad_consistency=1.4826) — и зачем это сделали

mad_consistency: float = 1.4826 - это Константа — коэффициент “приведения к сопоставимой шкале со стандартным отклонением” при нормальном распределении. У стандартной нормали медиана |Z| ≈ 0.67449, поэтому σ ≈ MAD / 0.67449 ≈ 1.4826 × MAD.
## Зачем:
- Сделать штрафы RPCA сопоставимыми между активами (без этого высоковолатильные активы доминируют в оптимизации).
- Упростить извлечение структуры: RPCA лучше выделяет низкоранговую часть (общерыночные/секторные компоненты) и разрежённые всплески, когда колонкам придан схожий масштаб.
- Сохраняем обратимость: благодаря scale_vec (и ранее сохранённому shift_vec) можно вернуться в исходные единицы после RPCA, чтобы корректно строить ковариации и портфели.

# Step 3: RPCA (ADMM или FISTA)

In [20]:
%run robust_PCA_step3.py

LinAlgError: SVD did not converge